this notebook is build to walk through the




before running imports we set up a venv this is simply done by running the following lines one by one in the terminal:

mac:

#############################################

python -m venv venv

source venv/bin/activate  

pip install -e .

#############################################

windows:

#############################################

python -m venv venv

venv\Scripts\activate

pip install -e .

#############################################



then select the venv as juniper kernel




In [1]:
import numpy as np
import pandas as pd
import datetime as dt
from scipy.stats import norm
from src.pricing.heston_model import HestonModel
from src.pricing.fourier_pricing import bsm_cf, compute_lewis_strikes_quad,compute_bsm
import argparse
from src.scripts.calibrate import load_data,build_target,run_calibrations,compute_iv_surfaces,plot_iv_surfaces,fit_rates
from src.scripts.benchmark_calibration import experiment



## Fourier pricing — BSM sanity check

In [2]:
#we first check our lewis pricing implimentation againt the BSM fomula


s_0, k, r, t, sigma, q = 100.0, 100.0, 0.05, 1.0, 0.2, 0
cf = bsm_cf(sigma,s_0,t,r,q)
strikes = [80, 90, 100,110, 120, 130, 140, 150, 160, 170, 180]
lewis_prices = compute_lewis_strikes_quad(cf,s_0,t,strikes,r,q,)
bsm_prices = np.array([compute_bsm(s_0,k,r,t,sigma,q) for k in strikes])
print(sum(np.abs(lewis_prices-bsm_prices)))






1.537826471587067e-05


acros 11 maturities we get a total sum of differences to be around 1/10^5, i would say that seems reasonable

## Heston model — simulation

In [3]:
seed = 42
# we simulate 1 milion paths and plot 5 of them
# the seeding is a little funky, to get the exact picture use    
# S0=100.0, v0=0.04, r=0.05,q=0.02,
#    kappa=2.0, theta=0.04, sigma=0.6, rho=-0.7,
#    T=1.0, N=512, n_paths=1000000, seed=42



model = HestonModel(
    S0=100.0, v0=0.04, r=0.05,q=0.02,
    kappa=2.0, theta=0.04, sigma=0.6, rho=-0.7,
    T=1.0, N=512, n_paths=1000000, seed=seed)
strike =100
model.create_simulation()
model.plot_simulation()


100%|██████████| 512/512 [00:12<00:00, 41.99it/s]


In [4]:
#to evaluate the lewis pricing implimentation we compare implied volatilities generated by our pricing 
#with implied volatilities generated by the simulation we just did

strikes = [80,90, 100,110, 120,130 ,140,150, 160,170, 180]
print([model.get_simulated_price(k) for k in strikes])
print(f"lewis prices (vectorized): {model.compute_lewis_heston_prices(strikes)}")
model.plot_iv(strikes)

[np.float64(23.453460787058027), np.float64(15.428515992289789), np.float64(8.571178703924886), np.float64(3.612317787355804), np.float64(1.0763344797102496), np.float64(0.2655076062701598), np.float64(0.06778609118656395), np.float64(0.019372296063339275), np.float64(0.006234758260039242), np.float64(0.002238953056263664), np.float64(0.0008222307973155272)]
lewis prices (vectorized): [2.34532640e+01 1.54300277e+01 8.57339323e+00 3.61027279e+00
 1.07067719e+00 2.63155774e-01 6.67502964e-02 1.86428143e-02
 5.71708566e-03 1.90015016e-03 6.76826284e-04]


whih looks basically spot on as we have deviations which are at most on the 4. deciamal

## Calibration speed experiment

In [5]:
seed=42
experiment(seed=seed)

Strikes    : [60, 70, 80, 90, 100, 110, 120, 130, 140]
Maturities : [0.0833, 0.1667, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
Grid       : 9 × 8 = 72 options per trial
S0=100.0,  r=0.03,  q=0.01,  noise=1%,  trials=20

      |              GK FD               |              GK AD               |             Trap FD              |             Trap AD             
Trial |  time(s) |    Σ|Δθ| |       cost |  time(s) |    Σ|Δθ| |       cost |  time(s) |    Σ|Δθ| |       cost |  time(s) |    Σ|Δθ| |       cost
-----+----------+----------+------------+----------+----------+------------+----------+----------+------------+----------+----------+------------
    1 |    5.046 |      --- |     0.0000 |    4.730 |   0.0000 |     0.0000 |    1.320 |   0.0000 |     0.0000 |    0.640 |   0.0000 |     0.0000
    2 |    3.158 |      --- |     0.0000 |    2.805 |   0.0000 |     0.0000 |    0.834 |   0.0000 |     0.0000 |    0.364 |   0.0000 |     0.0000
    3 |    3.197 |      --- |     0.0000 |    2.107 |   0.00

## Calibrating the model on real data (2019-11-13)

In [6]:
t_0 = dt.date(2019, 11, 13)
s0 = 3094

calls, puts = load_data()
r_series, q = fit_rates(puts, calls, s0, t_0)
target_df = build_target(calls, puts, s0, r_series, q, t_0)

print(r_series,q)

# we generate the markets implied volatility surface by commenting out both calibration runs
runs = {
    "unweighted":         None,
    "relative (% error)": lambda df: 1.0 / df["target"].values,
}

all_fitted = run_calibrations(target_df, s0, q, runs)
iv_df = compute_iv_surfaces(target_df, all_fitted, s0, q)
plot_iv_surfaces(iv_df, all_fitted)


#to hide a surface press it on the right

fitted q = 0.0191,  r range = [0.0163, 0.0231]
calibration set: 1200 options (363 OTM calls + 837 OTM puts via parity) across 11 maturities
2019-12-20    0.016340
2020-01-17    0.023147
2020-02-21    0.021392
2020-03-20    0.020627
2020-04-17    0.021285
2020-06-19    0.020035
2020-09-18    0.019338
2020-10-16    0.019783
2020-11-20    0.019298
2020-12-18    0.018946
2021-06-18    0.018789
2021-12-17    0.018620
dtype: float64 0.019096145035877164

calibration — unweighted:
`ftol` termination condition is satisfied.
Function evaluations 12, initial cost 2.8821e+05, final cost 2.1943e+03, first-order optimality 1.54e+00.
  params: v0: 0.014758371056669465, kappa: 2.7211785713205945, theta: 0.04954446487353723, sigma: 0.8777534867093126, rho: -0.8290213296226108
  cost:   2194.2809  time: 9.81s

calibration — relative (% error):
`ftol` termination condition is satisfied.
Function evaluations 12, initial cost 4.8103e+03, final cost 1.0287e+00, first-order optimality 4.45e-04.
  params: v0